# Week 6 – Feature Engineering and Market Metrics

**Purpose:** Create the engineered market metrics required for the IDX Exchange Week 6 deliverable and prepare output files for future Tableau dashboard development.

This notebook is organized into two sections:

1. **Sold data metrics** – used for closed-sales analysis, pricing ratios, days on market, and transaction timeline metrics.
2. **Listing data metrics** – used for new-listing activity, listing-side price per square foot, and market supply summaries.

**Main deliverables created in this notebook:**

- Engineered sold dataset
- Engineered listing dataset
- Sample output tables showing new columns populated correctly
- Segmented summary tables by county, property type/subtype, MLS area, and office

## 1. Setup: Import Packages and Define File Paths

This section imports the required Python packages, defines the project folder, identifies the input files from the previous weeks, and creates the Week 6 output folder.

> Update `SOLD_FILE` and `LISTINGS_FILE` only if your actual file names are different.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Base project directory
BASE_DIR = Path("/Users/amyliu/Desktop/IDX")

# Input files from previous cleaned / enriched outputs
SOLD_FILE = BASE_DIR / "data" / "generated" / "sold_with_rates_week4-5.csv"
LISTINGS_FILE = BASE_DIR / "data" / "generated" / "listing_with_rates_week4-5.csv"

# Week 6 output folder
OUTPUT_DIR = BASE_DIR / "data" / "generated" / "week6"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Sold input file:", SOLD_FILE)
print("Listings input file:", LISTINGS_FILE)
print("Output directory:", OUTPUT_DIR)

if not SOLD_FILE.exists():
    raise FileNotFoundError(f"Sold file not found: {SOLD_FILE}")

if not LISTINGS_FILE.exists():
    raise FileNotFoundError(f"Listings file not found: {LISTINGS_FILE}")

## 2. Load Input Datasets

The sold dataset is used to create closed-sales metrics. The listing dataset is used to create new-listing and market activity metrics.

In [ ]:
sold = pd.read_csv(SOLD_FILE, low_memory=False)

print("Sold dataset")
print(f"Rows loaded: {len(sold):,}")
print(f"Columns loaded: {sold.shape[1]:,}")

sold.head()

In [ ]:
listings = pd.read_csv(LISTINGS_FILE, low_memory=False)

print("Listing dataset")
print(f"Rows loaded: {len(listings):,}")
print(f"Columns loaded: {listings.shape[1]:,}")

listings.head()

---

# Part A – Sold Data Feature Engineering

The sold dataset supports the official Week 6 metrics such as price ratio, price per square foot, close-to-original-list ratio, and transaction timeline metrics.

## 3. Check Required Sold Columns

Before creating features, this step verifies whether the required sold-side columns exist in the dataset.

In [ ]:
sold_required_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName",
    "BuyerOfficeName"
]

sold_column_check = pd.DataFrame({
    "column": sold_required_cols,
    "exists": [col in sold.columns for col in sold_required_cols]
})

sold_column_check

## 4. Convert Sold Date and Numeric Fields

The transaction timeline metrics require date fields to be in datetime format. Price and area fields need to be numeric before ratios can be calculated.

In [ ]:
# Convert sold date columns
sold_date_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

for col in sold_date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors="coerce")
        print(f"Converted to datetime: {col}")
    else:
        print(f"Missing date column: {col}")

# Convert sold numeric columns
sold_numeric_cols = [
    "ClosePrice",
    "OriginalListPrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in sold_numeric_cols:
    if col in sold.columns:
        sold[col] = pd.to_numeric(sold[col], errors="coerce")
        print(f"Converted to numeric: {col}")
    else:
        print(f"Missing numeric column: {col}")

## 5. Define Safe Division Function

This helper function prevents invalid calculations when the denominator is missing or equal to zero.

In [ ]:
def safe_divide(numerator, denominator):
    """
    Return numerator / denominator.
    If denominator is missing or zero, return NaN.
    """
    return np.where(
        denominator.notna() & (denominator != 0),
        numerator / denominator,
        np.nan
    )

## 6. Create Sold-Side Week 6 Metrics

This step creates the official Week 6 sold-side engineered metrics.

| Metric | Formula / Source | Purpose |
|---|---|---|
| `price_ratio` | `ClosePrice / OriginalListPrice` | Measures negotiation strength |
| `close_to_original_list_ratio` | `ClosePrice / OriginalListPrice` | Captures full price reduction history |
| `price_per_sqft` | `ClosePrice / LivingArea` | Normalizes price across different home sizes |
| `days_on_market` | `DaysOnMarket` | Measures time-to-sell |
| `close_year`, `close_month`, `yrmo` | Derived from `CloseDate` | Enables time-series analysis |
| `listing_to_contract_days` | `PurchaseContractDate - ListingContractDate` | Measures time from listing to accepted offer |
| `contract_to_close_days` | `CloseDate - PurchaseContractDate` | Measures escrow / closing period duration |

In [ ]:
# Price Ratio = ClosePrice / OriginalListPrice
sold["price_ratio"] = safe_divide(
    sold["ClosePrice"],
    sold["OriginalListPrice"]
)

# Close-to-Original-List Ratio = ClosePrice / OriginalListPrice
sold["close_to_original_list_ratio"] = safe_divide(
    sold["ClosePrice"],
    sold["OriginalListPrice"]
)

# Price Per Sq Ft = ClosePrice / LivingArea
sold["price_per_sqft"] = safe_divide(
    sold["ClosePrice"],
    sold["LivingArea"]
)

# Days on Market
sold["days_on_market"] = sold["DaysOnMarket"]

# Year / Month / YrMo from CloseDate
sold["close_year"] = sold["CloseDate"].dt.year
sold["close_month"] = sold["CloseDate"].dt.month
sold["yrmo"] = sold["CloseDate"].dt.to_period("M").astype(str)

# Listing-to-Contract Days = PurchaseContractDate - ListingContractDate
sold["listing_to_contract_days"] = (
    sold["PurchaseContractDate"] - sold["ListingContractDate"]
).dt.days

# Contract-to-Close Days = CloseDate - PurchaseContractDate
sold["contract_to_close_days"] = (
    sold["CloseDate"] - sold["PurchaseContractDate"]
).dt.days

sold_engineered_cols = [
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "close_year",
    "close_month",
    "yrmo",
    "listing_to_contract_days",
    "contract_to_close_days"
]

sold[sold_engineered_cols].head()

## 7. Validate Sold Engineered Metrics

This section checks the missing-value pattern and distribution of the newly created sold-side metrics.

In [ ]:
sold_metric_nulls = (
    sold[sold_engineered_cols]
    .isna()
    .sum()
    .reset_index()
)

sold_metric_nulls.columns = ["engineered_column", "null_count"]
sold_metric_nulls["null_pct"] = sold_metric_nulls["null_count"] / len(sold)

sold_metric_nulls

In [ ]:
sold_metric_summary_cols = [
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "listing_to_contract_days",
    "contract_to_close_days"
]

sold_metric_summary = (
    sold[sold_metric_summary_cols]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])
    .T
)

sold_metric_summary

## 8. Sold Sample Output Table

This sample table demonstrates that the required sold-side engineered columns were created and populated correctly.

In [ ]:
sold_sample_cols = [
    "CloseDate",
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "close_year",
    "close_month",
    "yrmo",
    "ListingContractDate",
    "PurchaseContractDate",
    "listing_to_contract_days",
    "contract_to_close_days",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor"
]

sold_sample_cols_existing = [col for col in sold_sample_cols if col in sold.columns]
week6_sold_sample_output = sold[sold_sample_cols_existing].head(20)

week6_sold_sample_output

## 9. Sold Segmented Summary by County

This table satisfies the Week 6 requirement to include at least one segmented summary table grouped by `CountyOrParish` or `PropertyType`. It summarizes sales volume, prices, price per square foot, days on market, and transaction timeline metrics by county.

In [ ]:
county_summary = (
    sold.groupby("CountyOrParish", dropna=False)
    .agg(
        closed_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("days_on_market", "mean"),
        median_days_on_market=("days_on_market", "median"),
        average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean"),
        median_listing_to_contract_days=("listing_to_contract_days", "median"),
        median_contract_to_close_days=("contract_to_close_days", "median")
    )
    .reset_index()
    .sort_values("closed_sales", ascending=False)
)

county_summary.head(20)

## 10. Sold Segmented Summary by Property Type and Subtype

This table supports later Tableau filtering by property category and helps compare pricing and market speed across property subtypes.

In [ ]:
property_summary = (
    sold.groupby(["PropertyType", "PropertySubType"], dropna=False)
    .agg(
        closed_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("days_on_market", "mean"),
        average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean")
    )
    .reset_index()
    .sort_values("closed_sales", ascending=False)
)

property_summary.head(20)

## 11. Sold Segmented Summary by MLS Area

This table provides a geographic market summary at the MLS area level.

In [ ]:
mls_area_summary = (
    sold.groupby("MLSAreaMajor", dropna=False)
    .agg(
        closed_sales=("ClosePrice", "count"),
        median_close_price=("ClosePrice", "median"),
        average_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("price_per_sqft", "median"),
        average_days_on_market=("days_on_market", "mean"),
        average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean")
    )
    .reset_index()
    .sort_values("closed_sales", ascending=False)
)

mls_area_summary.head(20)

## 12. Sold Competitive Summary by Listing Office and Buyer Office

This summary supports later competitive intelligence work by identifying office-level closed sales volume and transaction value.

In [ ]:
office_summary = (
    sold.groupby(["ListOfficeName", "BuyerOfficeName"], dropna=False)
    .agg(
        closed_sales=("ClosePrice", "count"),
        total_sales_volume=("ClosePrice", "sum"),
        median_close_price=("ClosePrice", "median"),
        average_days_on_market=("days_on_market", "mean"),
        average_close_to_original_list_ratio=("close_to_original_list_ratio", "mean")
    )
    .reset_index()
    .sort_values("total_sales_volume", ascending=False)
)

office_summary.head(20)

---

# Part B – Listing Data Feature Engineering

The listing dataset is used for market activity metrics, especially **new listings**, which will be needed in Tableau dashboards. Listing records do not always have sold-side fields such as `ClosePrice`, so the listing-side features focus on `ListingContractDate`, `ListPrice`, `LivingArea`, and listing activity summaries.

## 13. Check Required Listing Columns

This step verifies whether the required listing-side columns exist in the dataset.

In [ ]:
listing_required_cols = [
    "ListingContractDate",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName",
    "City",
    "PostalCode"
]

listing_column_check = pd.DataFrame({
    "column": listing_required_cols,
    "exists": [col in listings.columns for col in listing_required_cols]
})

listing_column_check

## 14. Convert Listing Date and Numeric Fields

The listing-side metrics require `ListingContractDate`, `ListPrice`, `LivingArea`, and `DaysOnMarket` to be in the correct data types.

In [ ]:
# Convert listing date field
listings["ListingContractDate"] = pd.to_datetime(
    listings["ListingContractDate"],
    errors="coerce"
)

# Convert listing numeric fields
listing_numeric_cols = [
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in listing_numeric_cols:
    listings[col] = pd.to_numeric(listings[col], errors="coerce")
    print(f"Converted {col} to numeric")

## 15. Create Listing-Side Week 6 Metrics

Listing-side metrics focus on new-listing activity and listing price normalization.

| Metric | Formula / Source | Purpose |
|---|---|---|
| `list_year`, `list_month`, `list_yrmo` | Derived from `ListingContractDate` | Enables monthly new-listing analysis |
| `list_price_per_sqft` | `ListPrice / LivingArea` | Normalizes listing price across home sizes |
| `listing_days_on_market` | `DaysOnMarket` | Measures listing-side market exposure |

In [ ]:
# Year / Month / YrMo from ListingContractDate
listings["list_year"] = listings["ListingContractDate"].dt.year
listings["list_month"] = listings["ListingContractDate"].dt.month
listings["list_yrmo"] = listings["ListingContractDate"].dt.to_period("M").astype(str)

# List Price Per Sq Ft = ListPrice / LivingArea
listings["list_price_per_sqft"] = safe_divide(
    listings["ListPrice"],
    listings["LivingArea"]
)

# Listing Days on Market
listings["listing_days_on_market"] = listings["DaysOnMarket"]

listing_engineered_cols = [
    "list_year",
    "list_month",
    "list_yrmo",
    "list_price_per_sqft",
    "listing_days_on_market"
]

listings[listing_engineered_cols].head()

## 16. Monthly New Listings Summary

This summary supports the Tableau dashboard requirement for tracking new listings over time.

In [ ]:
monthly_new_listings = (
    listings.groupby("list_yrmo", dropna=False)
    .agg(
        new_listings=("ListingContractDate", "count"),
        median_list_price=("ListPrice", "median"),
        average_list_price=("ListPrice", "mean"),
        median_list_price_per_sqft=("list_price_per_sqft", "median"),
        average_listing_days_on_market=("listing_days_on_market", "mean")
    )
    .reset_index()
    .sort_values("list_yrmo")
)

monthly_new_listings.head(20)

## 17. Listing Segmented Summary by County

This table summarizes listing activity by county, including new-listing count, list price, list price per square foot, and listing days on market.

In [ ]:
listing_county_summary = (
    listings.groupby("CountyOrParish", dropna=False)
    .agg(
        new_listings=("ListingContractDate", "count"),
        median_list_price=("ListPrice", "median"),
        average_list_price=("ListPrice", "mean"),
        median_list_price_per_sqft=("list_price_per_sqft", "median"),
        average_listing_days_on_market=("listing_days_on_market", "mean")
    )
    .reset_index()
    .sort_values("new_listings", ascending=False)
)

listing_county_summary.head(20)

## 18. Listing Segmented Summary by Property Type and Subtype

This summary supports property-level market activity analysis for future Tableau filters.

In [ ]:
listing_property_summary = (
    listings.groupby(["PropertyType", "PropertySubType"], dropna=False)
    .agg(
        new_listings=("ListingContractDate", "count"),
        median_list_price=("ListPrice", "median"),
        average_list_price=("ListPrice", "mean"),
        median_list_price_per_sqft=("list_price_per_sqft", "median"),
        average_listing_days_on_market=("listing_days_on_market", "mean")
    )
    .reset_index()
    .sort_values("new_listings", ascending=False)
)

listing_property_summary.head(20)

## 19. Listing Sample Output Table

This sample table verifies that the listing-side engineered columns were created correctly.

In [ ]:
listing_sample_cols = [
    "ListingContractDate",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "list_year",
    "list_month",
    "list_yrmo",
    "list_price_per_sqft",
    "listing_days_on_market",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "City",
    "PostalCode",
    "ListOfficeName"
]

listing_sample_cols_existing = [col for col in listing_sample_cols if col in listings.columns]
week6_listing_sample_output = listings[listing_sample_cols_existing].head(20)

week6_listing_sample_output

---

# Part C – Save Week 6 Outputs

This section saves the engineered datasets, sample output tables, validation summaries, and segmented summary tables.

In [ ]:
# Sold outputs
sold_engineered_output = OUTPUT_DIR / "sold_week6_engineered_metrics.csv"
sold_sample_output = OUTPUT_DIR / "week6_sold_sample_output.csv"
county_output = OUTPUT_DIR / "week6_sold_county_summary.csv"
property_output = OUTPUT_DIR / "week6_sold_property_summary.csv"
mls_area_output = OUTPUT_DIR / "week6_sold_mls_area_summary.csv"
office_output = OUTPUT_DIR / "week6_sold_office_summary.csv"
sold_null_summary_output = OUTPUT_DIR / "week6_sold_metric_null_summary.csv"
sold_metric_summary_output = OUTPUT_DIR / "week6_sold_metric_summary.csv"

# Listing outputs
listing_engineered_output = OUTPUT_DIR / "listing_week6_engineered_metrics.csv"
listing_sample_output = OUTPUT_DIR / "week6_listing_sample_output.csv"
monthly_new_listings_output = OUTPUT_DIR / "week6_monthly_new_listings.csv"
listing_county_output = OUTPUT_DIR / "week6_listing_county_summary.csv"
listing_property_output = OUTPUT_DIR / "week6_listing_property_summary.csv"

# Save sold files
sold.to_csv(sold_engineered_output, index=False)
week6_sold_sample_output.to_csv(sold_sample_output, index=False)
county_summary.to_csv(county_output, index=False)
property_summary.to_csv(property_output, index=False)
mls_area_summary.to_csv(mls_area_output, index=False)
office_summary.to_csv(office_output, index=False)
sold_metric_nulls.to_csv(sold_null_summary_output, index=False)
sold_metric_summary.to_csv(sold_metric_summary_output)

# Save listing files
listings.to_csv(listing_engineered_output, index=False)
week6_listing_sample_output.to_csv(listing_sample_output, index=False)
monthly_new_listings.to_csv(monthly_new_listings_output, index=False)
listing_county_summary.to_csv(listing_county_output, index=False)
listing_property_summary.to_csv(listing_property_output, index=False)

print("Week 6 outputs saved to:", OUTPUT_DIR)
print("
Sold outputs:")
for path in [
    sold_engineered_output,
    sold_sample_output,
    county_output,
    property_output,
    mls_area_output,
    office_output,
    sold_null_summary_output,
    sold_metric_summary_output
]:
    print(path)

print("
Listing outputs:")
for path in [
    listing_engineered_output,
    listing_sample_output,
    monthly_new_listings_output,
    listing_county_output,
    listing_property_output
]:
    print(path)

## Week 6 Summary

In this notebook, I created the required Week 6 feature-engineered market metrics from both the sold and listing datasets.

For the **sold dataset**, I engineered price ratio, close-to-original-list ratio, price per square foot, days on market, year/month/YrMo variables, listing-to-contract days, and contract-to-close days. I also generated sample output and segmented summaries by county, property type/subtype, MLS area, and office.

For the **listing dataset**, I created listing-side time variables, list price per square foot, listing days on market, monthly new-listings summaries, and segmented summaries by county and property type/subtype.

These outputs prepare the cleaned MLS data for future Tableau dashboard development and market analysis.